In [0]:
from pyspark.sql import functions as F

In [0]:
bronze_stop_times = spark.table("bg_traffic.bg_traffic_bronze.gtfs_stop_times")
bronze_stop_times.display()

In [0]:
required_columns = {"trip_id", "arrival_time", "departure_time", "stop_id", "stop_sequence"}
missing_columns = required_columns - set(bronze_stop_times.columns)
if missing_columns:
    raise ValueError(f"GRESKA: Izvorni GTFS stop_times je promenio strukturu, postoje nedostajuce kolone!")

In [0]:
bronze_stop_times.printSchema()

In [0]:
bronze_stop_times.select([F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c) for c in bronze_stop_times.columns]).show()

### Casting

In [0]:
types_stop_times = bronze_stop_times.select(
    F.col("trip_id").cast("string"),
    F.col("arrival_time").cast("string"),
    F.col("departure_time").cast("string"),
    F.col("stop_id").cast("integer"),
    F.col("stop_sequence").cast("integer")
)
types_stop_times.printSchema()

### Dedup

In [0]:
dedup_stop_times = types_stop_times.dropDuplicates(["trip_id","stop_sequence"])

dedup_count = types_stop_times.count() - dedup_stop_times.count()
print(f"Broj duplikata: {dedup_count}")


In [0]:
def normalize_gtfs_time(column):
    h = F.lpad(((F.split(F.col(column), ":")[0].cast("int") % 24).cast("string")), 2, "0")
    m = F.split(F.col(column), ":")[1]
    s = F.split(F.col(column), ":")[2]
    return F.concat_ws(":", h, m, s)

### Valid

In [0]:
valid_stop_times = dedup_stop_times.filter(
    (F.col("trip_id").isNotNull()) &
    (F.col("stop_sequence").isNotNull()) &
    (F.col("stop_id").isNotNull())
).withColumn(
    "arrival_time", normalize_gtfs_time("arrival_time")
       
).withColumn(
    "departure_time", normalize_gtfs_time("departure_time")
).withColumn(
    "silver_processed_at", F.current_timestamp()
)


valid_stop_times.display()

In [0]:
if valid_stop_times.isEmpty():
    raise Exception("GRESKA: Silver tabela za upisivanje je prazna nakon ciscenja!")

In [0]:
valid_stop_times.write.format("delta").mode("overwrite").option(
    "overwriteSchema","true"
).saveAsTable("bg_traffic.bg_traffic_silver.gtfs_stop_time")